# LGBM & XGBM Assignment
This notebook compares LightGBM and XGBoost models using the Titanic dataset.

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier


In [ ]:
# Load Dataset
df = pd.read_csv("Titanic.csv")

# Display first 5 rows
df.head()


In [ ]:
# Dataset Information
df.info()


In [ ]:
# Statistical Summary
df.describe(include='all')


In [ ]:
# Check Missing Values
df.isnull().sum()


In [ ]:
# Visualize Missing Values
plt.figure(figsize=(8,5))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.show()


In [ ]:
# Histograms
df.hist(figsize=(12,10))
plt.show()


In [ ]:
# Boxplots
plt.figure(figsize=(12,6))
sns.boxplot(data=df.select_dtypes(include=np.number))
plt.xticks(rotation=90)
plt.show()


In [ ]:
# Survival Count Plot
sns.countplot(x='Survived', data=df)
plt.title("Survival Count")
plt.show()


In [ ]:
# Survival by Gender
sns.barplot(x='Sex', y='Survived', data=df)
plt.title("Survival by Gender")
plt.show()


In [ ]:
# Handling Missing Values
num_imputer = SimpleImputer(strategy='mean')

for col in df.select_dtypes(include=np.number).columns:
    df[col] = num_imputer.fit_transform(df[[col]])

cat_imputer = SimpleImputer(strategy='most_frequent')

for col in df.select_dtypes(include='object').columns:
    df[col] = cat_imputer.fit_transform(df[[col]]).ravel()


In [ ]:
# Encode Categorical Variables
le = LabelEncoder()

for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col])


In [ ]:
# Features and Target
X = df.drop('Survived', axis=1)
y = df['Survived']


In [ ]:
# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


In [ ]:
# LightGBM Model
lgbm = LGBMClassifier()

lgbm.fit(X_train, y_train)

y_pred_lgbm = lgbm.predict(X_test)

print("LightGBM Accuracy:", accuracy_score(y_test, y_pred_lgbm))
print(classification_report(y_test, y_pred_lgbm))


In [ ]:
# XGBoost Model
xgbm = XGBClassifier(use_label_encoder=False, eval_metric='logloss')

xgbm.fit(X_train, y_train)

y_pred_xgbm = xgbm.predict(X_test)

print("XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgbm))
print(classification_report(y_test, y_pred_xgbm))


In [ ]:
# Confusion Matrix - LightGBM
cm_lgbm = confusion_matrix(y_test, y_pred_lgbm)

plt.figure(figsize=(5,4))
sns.heatmap(cm_lgbm, annot=True, fmt='d', cmap='Blues')

plt.title("LightGBM Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()


In [ ]:
# Confusion Matrix - XGBoost
cm_xgbm = confusion_matrix(y_test, y_pred_xgbm)

plt.figure(figsize=(5,4))
sns.heatmap(cm_xgbm, annot=True, fmt='d', cmap='Greens')

plt.title("XGBoost Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()


In [ ]:
# Cross Validation Scores
lgbm_scores = cross_val_score(lgbm, X, y, cv=5)
xgbm_scores = cross_val_score(xgbm, X, y, cv=5)

print("LightGBM CV Score:", lgbm_scores.mean())
print("XGBoost CV Score:", xgbm_scores.mean())


In [ ]:
# Comparison Table
comparison = pd.DataFrame({
    'Model': ['LightGBM', 'XGBoost'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_lgbm),
        accuracy_score(y_test, y_pred_xgbm)
    ]
})

print(comparison)


# Comparative Analysis

## LightGBM
- Faster training speed
- Efficient with large datasets
- Lower memory usage

## XGBoost
- Higher accuracy in many cases
- Better regularization
- More robust against overfitting

## Conclusion
Both LightGBM and XGBoost performed well on the Titanic dataset.
XGBoost may provide slightly better accuracy, while LightGBM is generally faster and more memory efficient.
